# Soil vs Building - XGBoost sub-classifier

Splits the `ClassID == 5` ("Bare Soil" in the Sentinel-2 scene classification) population
into **Soil (1)** and **Building (0)**, replacing the single hardcoded rule that does the
job today in `expand_class`:

```python
# server/inference/inference.py:39-42  <- what this model replaces
soil_mask = df['ClassID'] == 5
df.loc[soil_mask & (df['NDBI'] > 0.0), 'classifier'] = 'Building'
df.loc[soil_mask & (df['NDBI'] <= 0.0), 'classifier'] = 'Soil'
```

## How this differs from `model-train-urban.ipynb`

That notebook trains the same label on **22 features, including B05, B06, B07, B8A and
B09**. The production composite never downloads those bands - `EVALSCRIPT_S2` in
`server/bimonthly_composite.py` requests exactly seven:

```
B01, B02, B03, B04, B08, B11, B12      (each / 10000.0 -> reflectance 0..1)
```

so a model trained on 22 features cannot be run by the pipeline at all. This notebook is
restricted to **19 features that are all derivable from those seven**, the same way
`capstone_model_v2` is (7 bands + 5 indices = its 12-wide feature vector).

## Two traps this notebook is built around

1. **`BSI` means two different things.** The pipeline computes `(B03 + B08) / (B03 - B08)`,
   an unbounded quantity that is *not* the Bare Soil Index; the dataset's `BSI` column is the
   standard normalised one. On the first dataset row they are `-2.78` and `+0.0766`. Because
   `ensemble_predict` puts its own `BSI` column on the frame **before** `expand_class` runs, a
   feature named `BSI` would silently be fed the wrong numbers at inference. This model's
   feature is therefore named **`BSI_N`**.

2. **`Longitude` / `Latitude` are UTM metres, not degrees** (`277555.0`, `2639715.0`, stepping
   by 10 m). A spatial split keyed on `0.05` degrees would put every pixel in its own block and
   silently degrade to a random split. `BLOCK_SIZE_M` is in **metres**.

## What it produces

`soil_building_v1/` - laid out like `capstone_model_v2/` so the pipeline loads it the same
way, plus the `metadata.json` that `capstone_model_v2` lacks (its feature order lives only
inside `inference.py`, which is how the 22-vs-12 mismatch went unnoticed in the first place).

Runs on Kaggle CPU or GPU, no internet required.

## 1. Config

In [ ]:
# ------------------------------------------------------------------ data ----
# Tried in order. If none exist, the loader scans /kaggle/input for any csv/parquet
# whose header contains TARGET - so the dataset is found whatever the slug is called.
DATA_CANDIDATES = [
    "/kaggle/input/datasets/mezbaussalaheen/bare-soil-ground-truth-dataset/soil-dataset.parquet",
    "/kaggle/input/bare-soil-ground-truth-dataset/soil-dataset.parquet",
]

TARGET   = "soilClassifiedId"          # 1 = soil, 0 = building
POSITIVE = 1                           # which raw value means "soil"

# Only Sentinel-2 SCL "Bare Soil" pixels are this model's business. Everything else is
# already resolved upstream: ClassID 4 -> Tree/Crop, ClassID 6 -> Water.
CLASS_ID_COL    = "ClassID"
CLASS_ID_FILTER = 5

# Pretty names, keyed by the RAW values in TARGET.
CLASS_NAMES = {0: "Building", 1: "Soil"}

# Never enter the feature matrix. ClassID is the annotation the target was derived from
# (leakage); the coordinates let a tree memorise map tiles instead of learning spectra.
DROP_COLS = ["Longitude", "Latitude", "ClassID", "ClassName"]

# ----------------------------------------------------------------- split ----
# "spatial_block" - whole coordinate grid-cells go to one split only. Neighbouring 10 m
#                   pixels are near-duplicates, so this is the HONEST estimate.
# "stratified"    - random split. Comparable to model-train-urban.ipynb, but optimistic.
SPLIT_STRATEGY = "spatial_block"
BLOCK_SIZE_M   = 1000.0      # METRES - the coordinates are UTM, not degrees
TEST_SIZE      = 0.15
VAL_SIZE       = 0.15        # taken from what remains after the test split
SEED           = 42

# ----------------------------------------------------------------- model ----
USE_GPU            = True    # falls back to CPU automatically when unavailable
USE_SCALER         = True    # StandardScaler, fit on train only - capstone_model_v2 parity
NUM_BOOST_ROUND    = 5000    # the urban run ended at 2999/3000, still improving
EARLY_STOP_ROUNDS  = 100
BALANCE_CLASSES    = True    # scale_pos_weight
TUNE_THRESHOLD     = True    # maximise F1 on validation only

EARLY_STOP_METRIC = "aucpr"
EXTRA_METRICS     = ["logloss", "auc"]

# Same values as model-train-urban.ipynb, which already agree with capstone_model_v2's
# XGBoost member (max_depth 6, lr 0.05, subsample 0.8, colsample 0.8, seed 42, hist).
PARAMS = {
    "max_depth":        6,
    "eta":              0.05,          # learning rate
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "gamma":            0.0,
    "lambda":           1.0,           # L2
    "alpha":            0.0,           # L1
    "seed":             SEED,
}

# -------------------------------------------------------------- artifacts ----
ART_DIR     = "/kaggle/working/soil_building_v1"
BEST_MODEL  = "best_model.json"        # written mid-training on every improvement
FINAL_MODEL = "final_model.json"       # the sliced booster, written at the end
MODEL_VERSION = "soil_building_v1"

## 2. Imports and data loading

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve,
                             matthews_corrcoef)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

XGB_VERSION = tuple(int(x) for x in xgb.__version__.split(".")[:2])
print(f"xgboost {xgb.__version__}   scikit-learn {sklearn.__version__}   "
      f"pandas {pd.__version__}   numpy {np.__version__}")

Path(ART_DIR).mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

In [ ]:
def _read_any(p):
    return pd.read_parquet(p) if p.suffix.lower() in (".parquet", ".pq") else pd.read_csv(p)


def _has_target(p):
    """Peek at the header only - cheap even for large files."""
    try:
        if p.suffix.lower() in (".parquet", ".pq"):
            import pyarrow.parquet as pq
            return TARGET in pq.read_schema(p).names
        return TARGET in pd.read_csv(p, nrows=0).columns
    except Exception:
        return False


def load_dataset():
    for c in DATA_CANDIDATES:
        p = Path(c)
        if not p.exists():
            continue
        if not _has_target(p):
            print(f"skipping {p} (no '{TARGET}' column)")
            continue
        print(f"loading {p}")
        return _read_any(p)

    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError(
            "no /kaggle/input and none of DATA_CANDIDATES exist - attach the dataset "
            "or point DATA_CANDIDATES at the file")

    files = sorted(p for p in root.rglob("*")
                   if p.is_file() and p.suffix.lower() in (".csv", ".parquet", ".pq"))
    matches = [p for p in files if _has_target(p)]
    if not matches:
        found = "\n  ".join(str(p) for p in files[:25]) or "(nothing)"
        raise FileNotFoundError(
            f"no attached file has a '{TARGET}' column. Files seen:\n  {found}")

    folder = matches[0].parent
    shards = [p for p in matches if p.parent == folder]
    print(f"found {len(matches)} file(s) with '{TARGET}'; loading {len(shards)} from {folder}")
    for p in shards:
        print(f"  {p.name}")
    return pd.concat([_read_any(p) for p in shards], ignore_index=True)


df = load_dataset()

print(f"\nshape: {df.shape}")
print(f"columns: {list(df.columns)}")
assert TARGET in df.columns, f"'{TARGET}' not in the dataframe"
display(df.head())

## 2b. Keep only `ClassID == 5`

This model exists to break a tie that only happens inside the Sentinel-2 SCL "Bare Soil"
class. Rows from any other `ClassID` are a different problem and would teach it the wrong
decision boundary, so they are dropped here - before the split, before any feature is built.

The crosstab is printed **first**. Every sampled row of the source parquet shows
`ClassID = 5` paired with `soilClassifiedId = 1`, so it is worth confirming with your own
eyes that the *building* rows also live under `ClassID = 5` and not somewhere else. If the
filter leaves only one label the next cell stops with the counts, rather than training a
model that has nothing to learn.

In [ ]:
print(f"rows before filtering: {len(df):,}\n")

if CLASS_ID_COL in df.columns:
    xtab = pd.crosstab(df[CLASS_ID_COL], df[TARGET], dropna=False)
    xtab.columns = [f"{TARGET}={c}  ({CLASS_NAMES.get(c, c)})" for c in xtab.columns]
    xtab.index.name = CLASS_ID_COL
    print(f"{CLASS_ID_COL} x {TARGET} crosstab")
    display(xtab)

    kept = df[df[CLASS_ID_COL] == CLASS_ID_FILTER]
    dropped = len(df) - len(kept)
    print(f"keeping {CLASS_ID_COL} == {CLASS_ID_FILTER}: "
          f"{len(kept):,} rows  (dropping {dropped:,}, {dropped / max(len(df), 1):.2%})")

    surviving = set(kept[TARGET].dropna().unique().tolist())
    assert len(kept) > 0, (
        f"no rows have {CLASS_ID_COL} == {CLASS_ID_FILTER}. "
        f"Values present: {sorted(df[CLASS_ID_COL].unique().tolist())}")
    assert len(surviving) >= 2, (
        f"\n\nSTOP - {CLASS_ID_COL} == {CLASS_ID_FILTER} leaves only {TARGET} in "
        f"{sorted(surviving)}; there is nothing to learn.\n"
        f"The crosstab above shows which {CLASS_ID_COL} values carry the other label.\n"
        f"Either the filter is wrong for this dataset, or '{TARGET}' is not the "
        f"soil-vs-building column you think it is.")

    df = kept.reset_index(drop=True)
else:
    print(f"WARNING - no '{CLASS_ID_COL}' column in this dataset; no filter applied.")

# rows with a missing label are unusable
n_bad = int(df[TARGET].isna().sum())
if n_bad:
    print(f"dropping {n_bad:,} rows with a missing {TARGET}")
    df = df[df[TARGET].notna()].reset_index(drop=True)

print(f"\nrows after filtering : {len(df):,}")
print(f"\n{TARGET} value counts (these are what the model will be trained on):")
vc = df[TARGET].value_counts(dropna=False).sort_index()
for v, n in vc.items():
    print(f"  {v}  {CLASS_NAMES.get(v, v):<10} {n:>12,}  ({n / len(df):6.2%})")

## 3. Feature engineering

Every index is **recomputed from the raw bands** by `build_features`. The dataset's own index
columns are used only to *verify* those formulas in the next cell - they are never fed to the
model. That is deliberate: the identical function has to run inside the pipeline at inference
time, where only the seven downloaded bands exist, so training on anything else would be
training on numbers the pipeline cannot reproduce.

### What is in, and why

| # | Feature | Formula | Note |
|---|---|---|---|
| 1-7 | `B01 B02 B03 B04 B08 B11 B12` | raw reflectance 0..1 | the only bands the composite fetches |
| 8 | `NDVI` | (B08-B04)/(B08+B04) | **bit-identical to the pipeline's** |
| 9 | `EVI` | 2.5(B08-B04)/(B08+6*B04-7.5*B02+1) | **bit-identical to the pipeline's** |
| 10 | `NDBI` | (B11-B08)/(B11+B08) | **bit-identical**; the feature the old rule thresholded |
| 11 | `MNDWI` | (B03-B11)/(B03+B11) | **bit-identical to the pipeline's** |
| 12 | `NDWI` | (B03-B08)/(B03+B08) | |
| 13 | `SAVI` | 1.5(B08-B04)/(B08+B04+0.5) | |
| 14 | `BSI_N` | ((B11+B04)-(B08+B02))/((B11+B04)+(B08+B02)) | the **standard** BSI - see trap 1 |
| 15 | `BI` | sqrt((B04^2+B03^2)/2) | brightness |
| 16 | `AWEI` | 4(B03-B11) - (0.25*B08 + 2.75*B12) | |
| 17 | `UI` | (B12-B08)/(B12+B08) | urban index |
| 18 | `IBI` | (NDBI - (SAVI+MNDWI)/2) / (NDBI + (SAVI+MNDWI)/2) | built-up index, **clipped** |
| 19 | `R_B11_B12` | B11/B12 | SWIR ratio, **clipped** |

### What is out, and why

* **`B05 B06 B07 B8A B09`** - never downloaded by `EVALSCRIPT_S2`. A model that uses them
  cannot be run by the pipeline. This is the single reason this notebook exists.
* **`NDMI`** - identically `-NDBI` (the dataset prints `0.003279` against `-0.003279`).
  Perfectly collinear, so it carries no information NDBI does not already have.
* **`Longitude` `Latitude`** - UTM metres. A tree splits on them and memorises which map
  tile is which class; accuracy looks wonderful and the model transfers nowhere.
* **`ClassID`** - the annotation `soilClassifiedId` was derived from. Leakage.

### Why two features are clipped

`IBI` and `R_B11_B12` both divide by a quantity that passes through zero, so they are
unbounded - on synthetic reflectance `IBI` reaches ±7e5. XGBoost's `hist` tree method buckets
each feature into 256 bins, so a handful of extreme values would squash every real value into
a single bin and destroy the feature. Clipping to a documented range fixes that; the bounds
are recorded in `metadata.json` so inference clips identically. The next cell reports what
fraction of rows actually hit the bounds.

`EVI` is deliberately **not** clipped - it is reproduced exactly as the pipeline computes it,
divide-by-zero guard included, so the two agree to the bit.

In [ ]:
# The seven bands EVALSCRIPT_S2 downloads, in the order the pipeline names them.
SOURCE_BANDS = ["B01", "B02", "B03", "B04", "B08", "B11", "B12"]

# Documented so metadata.json carries them and inference can be checked against them.
INDEX_FORMULAS = {
    "NDVI":      "(B08 - B04) / (B08 + B04)",
    "EVI":       "2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 1)",
    "NDBI":      "(B11 - B08) / (B11 + B08)",
    "MNDWI":     "(B03 - B11) / (B03 + B11)",
    "NDWI":      "(B03 - B08) / (B03 + B08)",
    "SAVI":      "1.5 * (B08 - B04) / (B08 + B04 + 0.5)",
    "BSI_N":     "((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))",
    "BI":        "sqrt((B04**2 + B03**2) / 2)",
    "AWEI":      "4*(B03 - B11) - (0.25*B08 + 2.75*B12)",
    "UI":        "(B12 - B08) / (B12 + B08)",
    "IBI":       "(NDBI - (SAVI + MNDWI)/2) / (NDBI + (SAVI + MNDWI)/2)   [clipped]",
    "R_B11_B12": "B11 / B12   [clipped]",
}

# Unbounded ratios - see the note above.
CLIP_BOUNDS = {"IBI": (-10.0, 10.0), "R_B11_B12": (0.0, 10.0)}

FEATURES = SOURCE_BANDS + list(INDEX_FORMULAS)      # 7 + 12 = 19


# --------------------------------------------------------------------------------
# Defined as a STRING, then exec'd. Section 13 writes this identical string into
# predict_soil_building.py, so the code that ships is the code that trained - not a
# copy of it. (inspect.getsource would work in Jupyter and break under nbconvert or a
# plain exec; this cannot.)
# --------------------------------------------------------------------------------
FEATURE_SOURCE = r'''
def _safe_div(num, den):
    """Element-wise divide using the production pipeline's exact guard.

    server/inference/inference.py writes np.where(denom != 0, a / denom, 0.0).
    The inner np.where only stops numpy evaluating the division at the zeros
    (which would warn); the returned values are identical.
    """
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64)
    return np.where(den != 0, num / np.where(den != 0, den, 1.0), 0.0)


def build_features(frame):
    """Build the 19-feature matrix from the seven downloaded Sentinel-2 bands.

    Parameters
    ----------
    frame : pd.DataFrame
        Must contain B01, B02, B03, B04, B08, B11, B12 as reflectance in 0..1
        (Sentinel Hub DN / 10000, which is what EVALSCRIPT_S2 already returns).
        Any other columns are ignored - in particular the pipeline's own 'BSI',
        which is a different quantity and must never reach this model.

    Returns
    -------
    pd.DataFrame
        float32, columns exactly FEATURES in order.
    """
    missing = [b for b in SOURCE_BANDS if b not in frame.columns]
    if missing:
        raise KeyError(f"missing required band columns: {missing}")

    b01, b02, b03, b04, b08, b11, b12 = (
        frame[b].to_numpy(dtype=np.float64) for b in SOURCE_BANDS)

    out = pd.DataFrame(index=frame.index)
    for name, col in zip(SOURCE_BANDS, (b01, b02, b03, b04, b08, b11, b12)):
        out[name] = col

    # --- the four the pipeline already computes, reproduced exactly ---
    out["NDVI"]  = _safe_div(b08 - b04, b08 + b04)
    out["EVI"]   = _safe_div(2.5 * (b08 - b04), b08 + 6.0 * b04 - 7.5 * b02 + 1.0)
    out["NDBI"]  = _safe_div(b11 - b08, b11 + b08)
    out["MNDWI"] = _safe_div(b03 - b11, b03 + b11)

    # --- the rest ---
    out["NDWI"]  = _safe_div(b03 - b08, b03 + b08)
    out["SAVI"]  = _safe_div(1.5 * (b08 - b04), b08 + b04 + 0.5)
    out["BSI_N"] = _safe_div((b11 + b04) - (b08 + b02), (b11 + b04) + (b08 + b02))
    out["BI"]    = np.sqrt((b04 ** 2 + b03 ** 2) / 2.0)
    out["AWEI"]  = 4.0 * (b03 - b11) - (0.25 * b08 + 2.75 * b12)
    out["UI"]    = _safe_div(b12 - b08, b12 + b08)

    mid  = (out["SAVI"].to_numpy() + out["MNDWI"].to_numpy()) / 2.0
    ndbi = out["NDBI"].to_numpy()
    out["IBI"] = _safe_div(ndbi - mid, ndbi + mid)

    out["R_B11_B12"] = _safe_div(b11, b12)

    for name, (lo, hi) in CLIP_BOUNDS.items():
        out[name] = np.clip(out[name].to_numpy(), lo, hi)

    return out[FEATURES].astype(np.float32)
'''

exec(FEATURE_SOURCE, globals())

print(f"FEATURES ({len(FEATURES)}): {FEATURES}")
assert len(FEATURES) == len(set(FEATURES)), "duplicate feature name"
for banned in ("B05", "B06", "B07", "B8A", "B09", "NDMI", "BSI",
               "Longitude", "Latitude", "ClassID", "ClassName"):
    assert banned not in FEATURES, (
        f"'{banned}' must not be a feature - the pipeline either cannot compute it, "
        f"computes it differently, or it leaks the label")
print("confirmed: no unavailable band, no NDMI, no bare 'BSI', no coordinates, no ClassID")

### 3a. Sanity checks before a single tree is built

Two things are asserted here, because both fail *silently* if they are wrong:

1. **Band scale.** The pipeline divides raw DN by 10000 and hands the model reflectance in
   `0..1`. If this dataset stored DN instead, every feature would be off by 10000x and the
   model would be useless in production while still scoring beautifully in this notebook.
2. **Formula agreement.** `build_features` is checked against the dataset's own stored index
   columns. This is what proves `BSI_N` is the formula the dataset used, and that nothing was
   mis-transcribed.

In [ ]:
# ---- 1. band scale ----
scale_report = df[SOURCE_BANDS].median()
print("median reflectance per band:")
print(scale_report.round(4).to_string())
assert scale_report.between(0.0, 1.0).all(), (
    f"\n\nSTOP - bands are not in 0..1 reflectance:\n{scale_report}\n"
    f"The pipeline feeds the model DN/10000. If this dataset holds raw DN, divide by "
    f"10000 before going further, or the model will be unusable in production.")
print("\nOK - bands are reflectance in 0..1, matching EVALSCRIPT_S2's / 10000.0")

# ---- 2. formula agreement against the dataset's own index columns ----
_n = min(len(df), 250_000)
_sample = df.iloc[:_n]
_built  = build_features(_sample)

# our name -> the dataset's column name for the same quantity
CHECK_AGAINST = {
    "NDVI": "NDVI", "EVI": "EVI", "NDBI": "NDBI", "MNDWI": "MNDWI",
    "NDWI": "NDWI", "SAVI": "SAVI", "BSI_N": "BSI", "BI": "BI", "AWEI": "AWEI",
}
print(f"\nrecomputed vs stored, on {_n:,} rows (tolerance 1e-4):")
_bad = []
for mine, theirs in CHECK_AGAINST.items():
    if theirs not in _sample.columns:
        print(f"  --   {mine:<10} (dataset has no '{theirs}' column to compare against)")
        continue
    d = float(np.nanmax(np.abs(_built[mine].to_numpy(dtype=np.float64)
                               - _sample[theirs].to_numpy(dtype=np.float64))))
    ok = d < 1e-4
    _bad.append(mine) if not ok else None
    print(f"  {'OK ' if ok else 'FAIL'}  {mine:<10} vs '{theirs}':  max|diff| = {d:.2e}")
assert not _bad, (
    f"\n\nSTOP - {_bad} disagree with the dataset's own columns. A formula is wrong; "
    f"fix it before training, or the model will see different numbers at inference.")

# ---- how often the clipped features actually hit their bounds ----
print("\nclipping:")
for name, (lo, hi) in CLIP_BOUNDS.items():
    raw = _built[name].to_numpy()
    print(f"  {name:<10} [{lo}, {hi}]  at lower bound {np.mean(raw <= lo):6.3%}, "
          f"at upper bound {np.mean(raw >= hi):6.3%}")

# ---- NDMI really is -NDBI, which is why it is dropped ----
if "NDMI" in _sample.columns:
    r = float(np.corrcoef(_sample["NDMI"], _built["NDBI"])[0, 1])
    print(f"\ncorr(dataset NDMI, NDBI) = {r:+.6f}  -> NDMI dropped as redundant")

del _built, _sample

In [ ]:
# ---- build the full feature matrix and the label ----
X = build_features(df)

raw_y = df[TARGET]
if pd.api.types.is_float_dtype(raw_y) and np.allclose(raw_y.dropna(), raw_y.dropna().round()):
    raw_y = raw_y.round().astype(np.int64)      # 1.0 / 0.0 -> 1 / 0

CLASSES      = sorted(pd.unique(raw_y).tolist())
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
LABEL_NAMES  = [str(CLASS_NAMES.get(c, f"class {c}")) for c in CLASSES]

assert NUM_CLASSES == 2, (
    f"expected a binary soil/building label, got {NUM_CLASSES} values: {CLASSES}")
IS_BINARY = True

# The positive class (the one scale_pos_weight, the threshold and 'binary' averaging all
# refer to) must be Soil, so the reported precision/recall are soil precision/recall.
POS_IDX = CLASS_TO_IDX[POSITIVE]
assert POS_IDX == 1, (
    f"'{POSITIVE}' ({CLASS_NAMES.get(POSITIVE)}) encodes to index {POS_IDX}, not 1; "
    f"every threshold and metric below assumes index 1 is the positive class")

y = raw_y.map(CLASS_TO_IDX).astype(np.int16)

print(f"X: {X.shape}    y: {y.shape}    ({X.memory_usage(deep=True).sum()/2**20:.0f} MB)")
print(f"raw values : {CLASSES}")
print(f"encoded as : {[CLASS_TO_IDX[c] for c in CLASSES]}  ->  {LABEL_NAMES}")
print(f"positive class: index {POS_IDX} = '{LABEL_NAMES[POS_IDX]}'")

counts = y.value_counts().sort_index()
for i, n in counts.items():
    print(f"  {LABEL_NAMES[i]:<12} idx {i}  {n:>12,}  ({n / len(y):6.2%})")
print(f"imbalance ratio (max/min): {counts.max() / max(counts.min(), 1):.2f}x")

n_nan = int(X.isna().sum().sum())
print(f"NaNs in X: {n_nan}" + ("  (XGBoost handles these natively)" if n_nan else ""))
n_inf = int(np.isinf(X.to_numpy()).sum())
assert n_inf == 0, f"{n_inf} infinite values in X - a divide-by-zero guard is missing"
print("no infinities in X")

## 4. Train / validation / test split

`spatial_block` is the default. Neighbouring pixels are 10 m apart and near-identical, so a
random split scores the model on near-copies of its own training rows - that is why the
22-feature urban notebook reports F1 = 0.9875. Whole 1 km blocks go to exactly one split
here, matching `BLOCK_M = 1000.0` in the tree and water notebooks. Expect a lower number,
and expect it to be the one that predicts real map quality.

**`BLOCK_SIZE_M` is in metres because the coordinates are UTM metres.** The cell prints the
block count and the mean rows per block; if you ever point this at a dataset in degrees, the
block count will equal the row count and the assertion below will catch it.

In [ ]:
def _blocks(frame):
    """Grid-cell id per row, from UTM coordinates in metres."""
    need = {"Longitude", "Latitude"}
    if not need <= set(frame.columns):
        raise ValueError("spatial_block needs Longitude/Latitude in the dataframe")
    gx = np.floor(frame["Longitude"].to_numpy() / BLOCK_SIZE_M).astype(np.int64)
    gy = np.floor(frame["Latitude"].to_numpy() / BLOCK_SIZE_M).astype(np.int64)
    return np.char.add(np.char.add(gx.astype(str), "_"), gy.astype(str))


def spatial_block_split(y, frame):
    groups = _blocks(frame)
    n_blocks = len(np.unique(groups))
    print(f"spatial blocks: {n_blocks:,} cells of {BLOCK_SIZE_M:g} m "
          f"({len(y) / max(n_blocks, 1):,.0f} rows per block on average)")
    assert n_blocks >= 20, (
        f"only {n_blocks} distinct blocks - too few to split by. Lower BLOCK_SIZE_M.")
    assert n_blocks < len(y) * 0.5, (
        f"\n\nSTOP - {n_blocks:,} blocks for {len(y):,} rows means roughly one block per "
        f"pixel, so this is a random split wearing a spatial costume. BLOCK_SIZE_M "
        f"({BLOCK_SIZE_M:g}) is almost certainly in the wrong unit for these coordinates.")

    idx = np.arange(len(y))
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    tr_idx, te_idx = next(gss.split(idx, y, groups))
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=rel_val, random_state=SEED)
    sub_tr, sub_va = next(gss2.split(tr_idx, y.iloc[tr_idx], groups[tr_idx]))
    return tr_idx[sub_tr], tr_idx[sub_va], te_idx, groups


def stratified_split(y):
    idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y,
                                      random_state=SEED)
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    tr_idx, va_idx = train_test_split(tr_idx, test_size=rel_val, stratify=y.iloc[tr_idx],
                                      random_state=SEED)
    return tr_idx, va_idx, te_idx, None


if SPLIT_STRATEGY == "spatial_block":
    tr_idx, va_idx, te_idx, groups = spatial_block_split(y, df)
elif SPLIT_STRATEGY == "stratified":
    tr_idx, va_idx, te_idx, groups = stratified_split(y)
    print("NOTE - random split over 10 m pixels. Neighbouring pixels are near-duplicates,")
    print("       so these scores are optimistic. 'spatial_block' is the honest one.")
else:
    raise ValueError(f"unknown SPLIT_STRATEGY: {SPLIT_STRATEGY}")

assert not (set(tr_idx) & set(va_idx)) and not (set(tr_idx) & set(te_idx)) \
       and not (set(va_idx) & set(te_idx)), "splits overlap"

if groups is not None:
    g_tr, g_va, g_te = set(groups[tr_idx]), set(groups[va_idx]), set(groups[te_idx])
    assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), (
        "a spatial block appears in more than one split - the split leaks")
    print(f"blocks: train {len(g_tr):,}  val {len(g_va):,}  test {len(g_te):,}  "
          f"(disjoint)")

X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

split_table = pd.DataFrame(
    {name: part.value_counts().reindex(range(NUM_CLASSES), fill_value=0)
     for name, part in (("train", y_tr), ("val", y_va), ("test", y_te))})
split_table.index = LABEL_NAMES
split_table.loc["total"] = split_table.sum()
split_table["train %"] = (split_table["train"] / max(split_table.loc["total", "train"], 1)
                          ).map("{:.2%}".format)
print(f"\nsplit strategy: {SPLIT_STRATEGY}")
display(split_table)

for nm, part in (("train", y_tr), ("val", y_va), ("test", y_te)):
    assert part.nunique() == 2, f"the {nm} split lost a class - lower BLOCK_SIZE_M"

## 5. Scaling and XGBoost setup

`capstone_model_v2` scales its features with a `StandardScaler` before every prediction
(`server/inference/inference.py:98`), so this model keeps the same shape - one
`scaler.joblib` beside the model, fit on the **training split only**. Gradient-boosted trees
are invariant to monotone rescaling, so this costs nothing in accuracy; it buys a model that
drops into the existing loading code without a special case.

Set `USE_SCALER = False` in section 1 to skip it. `metadata.json` records which way it was
trained, and the generated inference helper reads that flag, so the two cannot disagree.

In [ ]:
if USE_SCALER:
    scaler = StandardScaler().fit(X_tr)
    def _scale(frame):
        return pd.DataFrame(scaler.transform(frame), columns=FEATURES,
                            index=frame.index).astype(np.float32)
    print("StandardScaler fit on the TRAIN split only "
          f"({len(X_tr):,} rows, {scaler.n_features_in_} features)")
    print(pd.DataFrame({"mean": scaler.mean_, "scale": scaler.scale_},
                       index=FEATURES).round(4).to_string())
else:
    scaler = None
    def _scale(frame):
        return frame.astype(np.float32)
    print("no scaler - features go into XGBoost raw")

Xs_tr, Xs_va, Xs_te = _scale(X_tr), _scale(X_va), _scale(X_te)

In [ ]:
def gpu_available():
    if not USE_GPU:
        return False
    try:
        import subprocess
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20)
        return r.returncode == 0 and "GPU" in r.stdout
    except Exception:
        return False


on_gpu = gpu_available()
params = dict(PARAMS)

# device/tree_method spelling changed in xgboost 2.0
if XGB_VERSION >= (2, 0):
    params["tree_method"] = "hist"
    params["device"] = "cuda" if on_gpu else "cpu"
else:
    params["tree_method"] = "gpu_hist" if on_gpu else "hist"

params["objective"] = "binary:logistic"

n_pos = int((y_tr == 1).sum())            # index 1 = Soil, asserted in section 3
n_neg = int(len(y_tr) - n_pos)
if BALANCE_CLASSES:
    params["scale_pos_weight"] = n_neg / max(n_pos, 1)

# the metric used for early stopping must be LAST in the list
metrics = [m for m in EXTRA_METRICS if m != EARLY_STOP_METRIC] + [EARLY_STOP_METRIC]
params["eval_metric"] = metrics
MAXIMIZE = EARLY_STOP_METRIC not in ("logloss", "mlogloss", "error", "merror", "rmse", "mae")

dtrain = xgb.DMatrix(Xs_tr, label=y_tr, feature_names=FEATURES)
dval   = xgb.DMatrix(Xs_va, label=y_va, feature_names=FEATURES)
dtest  = xgb.DMatrix(Xs_te, label=y_te, feature_names=FEATURES)

print(f"objective         : binary:logistic")
print(f"class balance     : {n_neg:,} x '{LABEL_NAMES[0]}' / {n_pos:,} x '{LABEL_NAMES[1]}'"
      + (f"   scale_pos_weight={params['scale_pos_weight']:.4f}" if BALANCE_CLASSES else ""))
print(f"device            : {'GPU (cuda)' if on_gpu else 'CPU'}")
print(f"eval metrics      : {metrics}   early stop on '{EARLY_STOP_METRIC}' "
      f"(maximize={MAXIMIZE})")
print(f"params            : {json.dumps(params, default=str)}")

## 6. Best-weight checkpoint callback

XGBoost's `save_model` writes **every** tree built so far. Saving after training finishes
therefore stores the extra `EARLY_STOP_ROUNDS` trees that made validation *worse*, and a
reloaded booster does not skip them unless you remember to pass `iteration_range`.

Two guards, both carried over from `model-train-urban.ipynb`:

1. The callback saves the booster **at the moment the validation metric improves**, so the
   file on disk never contains trees past the best iteration.
2. After training the booster is sliced with `booster[:best_iteration + 1]` and saved again.

Section 8 reloads both and asserts they give identical predictions.

In [ ]:
class BestModelCheckpoint(xgb.callback.TrainingCallback):
    """Persist the booster every time the watched validation metric improves."""

    def __init__(self, path, eval_set, metric, maximize=True, verbose=True):
        self.path, self.eval_set, self.metric = str(path), eval_set, metric
        self.maximize, self.verbose = maximize, verbose
        self.best_score = -np.inf if maximize else np.inf
        self.best_iteration = -1
        self.n_saves = 0

    def _improved(self, score):
        return score > self.best_score if self.maximize else score < self.best_score

    def after_iteration(self, model, epoch, evals_log):
        try:
            score = float(evals_log[self.eval_set][self.metric][-1])
        except (KeyError, IndexError):
            return False
        if self._improved(score):
            self.best_score, self.best_iteration = score, epoch
            model.save_model(self.path)          # <- best weights hit disk here
            self.n_saves += 1
            if self.verbose and self.n_saves % 25 == 0:
                print(f"    [ckpt] iter {epoch:>5}  {self.metric}={score:.6f}"
                      f"  -> {Path(self.path).name}")
        return False   # never stop training; early stopping owns that


best_ckpt_path = Path(ART_DIR) / BEST_MODEL

## 7. Train

In [ ]:
def _fit(p):
    """One training run. Returns (booster, evals_result, callback)."""
    cb = BestModelCheckpoint(best_ckpt_path, "val", EARLY_STOP_METRIC, maximize=MAXIMIZE)
    ev = {}
    b = xgb.train(
        p,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, "train"), (dval, "val")],   # early stopping watches the LAST entry
        early_stopping_rounds=EARLY_STOP_ROUNDS,
        evals_result=ev,
        callbacks=[cb],
        verbose_eval=50,
    )
    return b, ev, cb


t0 = time.time()
try:
    booster, evals_result, checkpoint_cb = _fit(params)
except Exception as exc:
    if not on_gpu:
        raise
    # a GPU is visible but unusable (driver / build mismatch) - fall back rather than die
    print(f"\nGPU training failed ({type(exc).__name__}: {exc})")
    print("retrying on CPU ...\n")
    on_gpu = False
    if XGB_VERSION >= (2, 0):
        params["device"] = "cpu"
    else:
        params["tree_method"] = "hist"
    t0 = time.time()
    booster, evals_result, checkpoint_cb = _fit(params)
elapsed = time.time() - t0

best_iter = int(getattr(booster, "best_iteration", checkpoint_cb.best_iteration))

print(f"\ntrained in {elapsed:.1f}s on {'GPU' if on_gpu else 'CPU'}")
print(f"boosting rounds      : {booster.num_boosted_rounds()}")
print(f"best_iteration       : {best_iter}")
print(f"best val {EARLY_STOP_METRIC:<12}: {checkpoint_cb.best_score:.6f}")
print(f"checkpoint saves     : {checkpoint_cb.n_saves}")

assert best_iter == checkpoint_cb.best_iteration, (
    f"callback best ({checkpoint_cb.best_iteration}) != booster best ({best_iter})")
assert best_ckpt_path.exists(), "no checkpoint was written"

if best_iter >= NUM_BOOST_ROUND - 1:
    print(f"\nNOTE - best_iteration is the last round, so early stopping never fired and "
          f"the model was still improving when it ran out of rounds. Raise "
          f"NUM_BOOST_ROUND (or eta) and train again to get the rest of it.")

In [ ]:
# slice off the rounds built after the best iteration, then save
final_model_path = Path(ART_DIR) / FINAL_MODEL
best_booster = booster[: best_iter + 1]
best_booster.save_model(final_model_path)

print(f"full booster : {booster.num_boosted_rounds()} rounds")
print(f"sliced       : {best_booster.num_boosted_rounds()} rounds -> {final_model_path}")
print(f"best ckpt    : {best_ckpt_path} ({best_ckpt_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# training curves
fig, axes = plt.subplots(1, len(metrics), figsize=(5.5 * len(metrics), 4), squeeze=False)
for ax, m in zip(axes[0], metrics):
    ax.plot(evals_result["train"][m], label="train", lw=1.4)
    ax.plot(evals_result["val"][m], label="val", lw=1.4)
    ax.axvline(best_iter, color="crimson", ls="--", lw=1, label=f"best iter {best_iter}")
    ax.set_xlabel("boosting round"); ax.set_ylabel(m); ax.set_title(m)
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(Path(ART_DIR) / "training_curves.png", dpi=130, bbox_inches="tight")
plt.show()

## 8. Verify the checkpoint reloads correctly

In [ ]:
# A checkpoint you never reload is a checkpoint you can't trust.
reloaded = xgb.Booster()
reloaded.load_model(best_ckpt_path)

p_sliced   = best_booster.predict(dtest)
p_reloaded = reloaded.predict(dtest)
# the full booster needs an explicit range to ignore the post-best rounds
p_full_ok  = booster.predict(dtest, iteration_range=(0, best_iter + 1))
p_full_bad = booster.predict(dtest)   # what you'd get by forgetting iteration_range

print(f"reloaded rounds                      : {reloaded.num_boosted_rounds()}")
print(f"max |sliced - reloaded_checkpoint|   : {np.abs(p_sliced - p_reloaded).max():.3e}")
print(f"max |sliced - full[:best+1]|         : {np.abs(p_sliced - p_full_ok).max():.3e}")
print(f"max |sliced - full(no range)|        : {np.abs(p_sliced - p_full_bad).max():.3e}"
      f"  <- non-zero is the bug this guards against")

np.testing.assert_allclose(p_sliced, p_reloaded, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(p_sliced, p_full_ok, rtol=1e-5, atol=1e-6)
print("\nOK - the saved checkpoint reproduces the best model exactly")

model = reloaded   # everything below uses the reloaded checkpoint

## 9. Decision rule

The probability is `P(Soil)`. The cut-off is tuned to maximise F1 **on validation only** -
the test set stays untouched until the next section.

For reference the old hardcoded rule is `NDBI > 0 -> Building`, i.e. a fixed threshold on a
single feature. Section 10 scores both on the same test set, so the model has to earn its
place.

In [ ]:
p_val = model.predict(dval)

if TUNE_THRESHOLD:
    grid = np.linspace(0.05, 0.95, 181)
    f1s = [f1_score(y_va, (p_val >= t).astype(int), zero_division=0) for t in grid]
    THRESHOLD = float(grid[int(np.argmax(f1s))])
    print(f"tuned threshold : {THRESHOLD:.3f}   "
          f"(val F1 = {max(f1s):.4f}, vs {f1s[len(grid) // 2]:.4f} at 0.50)")
else:
    THRESHOLD = 0.5
    print("threshold fixed at 0.50")


def to_pred(proba):
    return (proba >= THRESHOLD).astype(int)


def as_matrix(proba):
    return np.column_stack([1.0 - proba, proba])

## 10. Test-set evaluation

In [ ]:
def evaluate(name, y_true, proba):
    pred = to_pred(proba)
    return {
        "split":     name,
        "n":         int(len(y_true)),
        "accuracy":  round(float(accuracy_score(y_true, pred)), 4),
        "precision": round(float(precision_score(y_true, pred, zero_division=0)), 4),
        "recall":    round(float(recall_score(y_true, pred, zero_division=0)), 4),
        "f1":        round(float(f1_score(y_true, pred, zero_division=0)), 4),
        "mcc":       round(float(matthews_corrcoef(y_true, pred)), 4),
        "roc_auc":   round(float(roc_auc_score(y_true, proba)), 4),
        "pr_auc":    round(float(average_precision_score(y_true, proba)), 4),
    }


p_train = model.predict(dtrain)
p_test  = model.predict(dtest)

results = pd.DataFrame([
    evaluate("train", y_tr, p_train),
    evaluate("val",   y_va, p_val),
    evaluate("test",  y_te, p_test),
])
print(f"positive class: '{LABEL_NAMES[1]}'   threshold: {THRESHOLD:.3f}   "
      f"split: {SPLIT_STRATEGY}")
display(results)

print("\nTEST classification report")
print(classification_report(y_te, to_pred(p_test), labels=[0, 1],
                            target_names=LABEL_NAMES, digits=4, zero_division=0))

gap = results.loc[0, "f1"] - results.loc[2, "f1"]
if gap > 0.05:
    print(f"NOTE - train F1 exceeds test F1 by {gap:.3f}; consider more regularisation "
          f"(max_depth, lambda, min_child_weight).")

### 10a. Does it actually beat the rule it replaces?

The rule in `expand_class` today is `NDBI > 0.0 -> Building`, so predicted-Soil is
`NDBI <= 0.0`. Scored on the same test rows, with the same metrics. If the model does not
clear this by a comfortable margin, it is not worth the extra 20 MB and the extra failure
mode - say so out loud rather than shipping it.

In [ ]:
ndbi_te  = X_te["NDBI"].to_numpy()          # unscaled - the rule uses the raw index
rule_pred = (ndbi_te <= 0.0).astype(int)    # NDBI <= 0 -> Soil (index 1)

comparison = pd.DataFrame([
    {"model": "NDBI > 0 rule (current pipeline)",
     "accuracy":  round(float(accuracy_score(y_te, rule_pred)), 4),
     "precision": round(float(precision_score(y_te, rule_pred, zero_division=0)), 4),
     "recall":    round(float(recall_score(y_te, rule_pred, zero_division=0)), 4),
     "f1":        round(float(f1_score(y_te, rule_pred, zero_division=0)), 4),
     "mcc":       round(float(matthews_corrcoef(y_te, rule_pred)), 4)},
    {"model": f"{MODEL_VERSION} (this notebook)",
     "accuracy":  round(float(accuracy_score(y_te, to_pred(p_test))), 4),
     "precision": round(float(precision_score(y_te, to_pred(p_test), zero_division=0)), 4),
     "recall":    round(float(recall_score(y_te, to_pred(p_test), zero_division=0)), 4),
     "f1":        round(float(f1_score(y_te, to_pred(p_test), zero_division=0)), 4),
     "mcc":       round(float(matthews_corrcoef(y_te, to_pred(p_test))), 4)},
])
display(comparison)

d_f1  = comparison.loc[1, "f1"]  - comparison.loc[0, "f1"]
d_mcc = comparison.loc[1, "mcc"] - comparison.loc[0, "mcc"]
print(f"model - rule:  F1 {d_f1:+.4f}   MCC {d_mcc:+.4f}")
if d_f1 <= 0:
    print("\nWARNING - the model does NOT beat the hardcoded rule on this test set. "
          "Do not wire it in; investigate the split, the label, or the features first.")
else:
    print(f"\nthe model beats the rule it replaces by {d_f1:.4f} F1")
RULE_COMPARISON = comparison.to_dict(orient="records")

In [ ]:
cm = confusion_matrix(y_te, to_pred(p_test), labels=[0, 1])
proba_te = as_matrix(p_test)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(NUM_CLASSES)); ax[0].set_yticks(range(NUM_CLASSES))
ax[0].set_xticklabels(LABEL_NAMES); ax[0].set_yticklabels(LABEL_NAMES)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, f"{v:,}", ha="center", va="center",
               color="white" if v > cm.max() / 2 else "black", fontsize=12)
ax[0].set_title(f"Confusion matrix (test), thr={THRESHOLD:.2f}")

yk = (np.asarray(y_te) == 1).astype(int)
fpr, tpr, _ = roc_curve(yk, p_test)
prec, rec, _ = precision_recall_curve(yk, p_test)
ax[1].plot(fpr, tpr, lw=1.7, label=f"{LABEL_NAMES[1]} (AUC={roc_auc_score(yk, p_test):.4f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8)
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].set_title("ROC (test)")
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)

ax[2].plot(rec, prec, lw=1.7,
           label=f"{LABEL_NAMES[1]} (AP={average_precision_score(yk, p_test):.4f})")
ax[2].scatter([recall_score(y_te, rule_pred)], [precision_score(y_te, rule_pred)],
              color="crimson", zorder=5, s=45, label="NDBI > 0 rule")
ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision")
ax[2].set_title("Precision-Recall (test)")
ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(Path(ART_DIR) / "evaluation.png", dpi=130, bbox_inches="tight")
plt.show()

per_class = pd.DataFrame({
    "class":   LABEL_NAMES,
    "raw_id":  [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
    "support": cm.sum(axis=1),
    "correct": np.diag(cm),
})
per_class["recall"] = (per_class["correct"] / per_class["support"].replace(0, np.nan)).round(4)
display(per_class)

## 11. Feature importance

In [ ]:
gain   = model.get_score(importance_type="gain")
weight = model.get_score(importance_type="weight")
cover  = model.get_score(importance_type="cover")

imp = pd.DataFrame({
    "feature": FEATURES,
    "gain":    [gain.get(f, 0.0)   for f in FEATURES],
    "weight":  [weight.get(f, 0.0) for f in FEATURES],
    "cover":   [cover.get(f, 0.0)  for f in FEATURES],
}).sort_values("gain", ascending=False).reset_index(drop=True)
imp["gain_pct"] = (imp["gain"] / max(imp["gain"].sum(), 1e-12) * 100).round(2)
display(imp)

top = imp.iloc[::-1]
plt.figure(figsize=(8, max(4, 0.34 * len(top))))
plt.barh(top["feature"], top["gain"], color="#2b6cb0")
plt.xlabel("gain"); plt.title("Feature importance (gain)"); plt.grid(axis="x", alpha=.3)
plt.tight_layout()
plt.savefig(Path(ART_DIR) / "feature_importance.png", dpi=130, bbox_inches="tight")
plt.show()

for banned in DROP_COLS:
    assert banned not in imp["feature"].values, f"'{banned}' leaked into the model"
print(f"confirmed: {DROP_COLS} are absent from the model")

unused = imp.loc[imp["gain"] == 0.0, "feature"].tolist()
if unused:
    print(f"unused features (zero gain, safe to drop next run): {unused}")

## 12. Save artifacts

Laid out like `capstone_model_v2/` so `_load_artifacts` in `server/inference/inference.py`
loads it with the same two lines it already uses, plus the one thing `capstone_model_v2`
does not have: **`metadata.json`**, carrying the feature order, the threshold and the index
formulas. `capstone_model_v2`'s feature order lives only inside `inference.py`, which is
exactly how a model trained on 22 features came to sit next to a pipeline that computes 12.

In [ ]:
art = Path(ART_DIR)

# --- the model, both ways ---
joblib.dump(best_booster, art / "xgboost.joblib")      # loads like every capstone_model_v2 file
# best_model.json / final_model.json are already on disk from sections 6-7

# --- the scaler ---
if scaler is not None:
    joblib.dump(scaler, art / "scaler.joblib")

# --- everything needed to reproduce the feature vector at inference ---
metadata = {
    "model_version":     MODEL_VERSION,
    "trained_at":        time.strftime("%Y-%m-%d %H:%M:%S"),
    "purpose":           ("binary soil-vs-building sub-classifier for ClassID == 5 pixels; "
                          "replaces the 'NDBI > 0.0' rule in expand_class"),
    "replaces":          "server/inference/inference.py expand_class, soil branch",

    "xgboost_version":   xgb.__version__,
    "sklearn_version":   sklearn.__version__,
    "joblib_version":    joblib.__version__,
    "device":            "cuda" if on_gpu else "cpu",

    "task":              "binary",
    "target":            TARGET,
    "class_id_filter":   {"column": CLASS_ID_COL, "value": CLASS_ID_FILTER},
    "idx_to_class":      {str(i): int(IDX_TO_CLASS[i]) for i in range(NUM_CLASSES)},
    "class_names":       {str(i): LABEL_NAMES[i] for i in range(NUM_CLASSES)},
    "positive_class":    {"index": 1, "raw": int(POSITIVE), "name": LABEL_NAMES[1]},
    "threshold":         round(float(THRESHOLD), 4),

    "features":          FEATURES,
    "n_features":        len(FEATURES),
    "source_bands":      SOURCE_BANDS,
    "index_formulas":    INDEX_FORMULAS,
    "clip_bounds":       {k: list(v) for k, v in CLIP_BOUNDS.items()},
    "band_scale":        "reflectance 0..1 (Sentinel Hub DN / 10000, as EVALSCRIPT_S2 returns)",
    "excluded_columns":  DROP_COLS,
    "excluded_bands":    ["B05", "B06", "B07", "B8A", "B09"],
    "excluded_reason":   "not downloaded by EVALSCRIPT_S2; the pipeline cannot compute them",
    "uses_scaler":       bool(scaler is not None),

    "split_strategy":    SPLIT_STRATEGY,
    "block_size_m":      (BLOCK_SIZE_M if SPLIT_STRATEGY == "spatial_block" else None),
    "split_sizes":       {"train": int(len(y_tr)), "val": int(len(y_va)),
                          "test": int(len(y_te))},

    "params":            {k: (v if isinstance(v, (int, float, str, list)) else str(v))
                          for k, v in params.items()},
    "num_boost_round":   NUM_BOOST_ROUND,
    "early_stop_rounds": EARLY_STOP_ROUNDS,
    "early_stop_metric": EARLY_STOP_METRIC,
    "best_iteration":    int(best_iter),
    "best_val_" + EARLY_STOP_METRIC: round(float(checkpoint_cb.best_score), 6),

    "metrics":           results.to_dict(orient="records"),
    "baseline_comparison": RULE_COMPARISON,
    "top_features":      imp.head(10)[["feature", "gain_pct"]].to_dict(orient="records"),
}

(art / "metadata.json").write_text(json.dumps(metadata, indent=2))
imp.to_csv(art / "feature_importance.csv", index=False)
results.to_csv(art / "metrics.csv", index=False)
per_class.to_csv(art / "per_class_test.csv", index=False)
comparison.to_csv(art / "baseline_comparison.csv", index=False)

print(f"artifacts in {art}:")
for p in sorted(art.iterdir()):
    print(f"  {p.name:<28}{p.stat().st_size / 1024:>10.1f} KB")

## 13. The hand-off module

`predict_soil_building.py` is written out as a **real file**, containing the exact
`FEATURE_SOURCE` string that section 3 exec'd to build the training features. What
ships is not a copy of the training code - it is the same string, so the two cannot
drift apart.

To wire it into the pipeline later, drop `soil_building_v1/` next to `capstone_model_v2/`
and replace the two threshold lines in `expand_class`:

```python
# server/inference/inference.py - before
soil_mask = df['ClassID'] == 5
df.loc[soil_mask & (df['NDBI'] > 0.0), 'classifier'] = 'Building'
df.loc[soil_mask & (df['NDBI'] <= 0.0), 'classifier'] = 'Soil'

# after
soil_mask = df['ClassID'] == 5
if soil_mask.any():
    df.loc[soil_mask, 'classifier'] = classify_soil_rows(df.loc[soil_mask])
```

`soil_mask` already isolates exactly the right pixels, and `classify_soil_rows` recomputes
its own features from the raw bands - so it does not matter whether the caller has already
added its own `NDVI`/`NDBI`/`BSI` columns. That matters: `ensemble_predict` puts a **different**
`BSI` on the frame, and this model never touches it.

One thing to know before expecting a visible change: `Building` is merged straight back into
`Soil` downstream (`LAND_COVER_MERGE = {"Building": "Soil"}` at `server/api_server.py:76`, and
`CLASS_MAP_CODES["Building"] = 3`, the same code as `Soil`). Until those, the colour ramps,
the Flutter legend and `ANALYZE_CACHE_KIND` are updated, a better sub-classifier changes
nothing the user can see.

In [ ]:
_HEADER = [
    "# ---------------------------------------------------------------------------",
    "# Soil-vs-building sub-classifier - generated by new_training_soil_building.ipynb.",
    "# Do not edit by hand: build_features below is the exact FEATURE_SOURCE string the",
    "# model was trained with, so training and inference cannot drift apart.",
    "#",
    "# Usage inside expand_class (server/inference/inference.py):",
    "#     soil_mask = df['ClassID'] == 5",
    "#     if soil_mask.any():",
    "#         df.loc[soil_mask, 'classifier'] = classify_soil_rows(df.loc[soil_mask])",
    "# ---------------------------------------------------------------------------",
    "import json",
    "import os",
    "",
    "import numpy as np",
    "import pandas as pd",
    "import joblib",
    "import xgboost as xgb",
    "",
    "_MODEL_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "
    + repr(MODEL_VERSION) + ")",
    "_ARTIFACTS = None",
    "",
    "",
    "def _load():",
    "    # Loaded once and reused - ensemble_predict runs on every classified tile.",
    "    global _ARTIFACTS",
    "    if _ARTIFACTS is None:",
    "        with open(os.path.join(_MODEL_DIR, 'metadata.json')) as fh:",
    "            meta = json.load(fh)",
    "        booster = xgb.Booster()",
    "        booster.load_model(os.path.join(_MODEL_DIR, 'best_model.json'))",
    "        scaler = (joblib.load(os.path.join(_MODEL_DIR, 'scaler.joblib'))",
    "                  if meta['uses_scaler'] else None)",
    "        _ARTIFACTS = {'meta': meta, 'booster': booster, 'scaler': scaler}",
    "    return _ARTIFACTS",
    "",
    "",
]

_CONST = [
    "SOURCE_BANDS = " + repr(SOURCE_BANDS),
    "FEATURES = " + repr(FEATURES),
    "CLIP_BOUNDS = " + repr(CLIP_BOUNDS),
    "THRESHOLD = " + repr(round(float(THRESHOLD), 4)),
    "SOIL_LABEL, BUILDING_LABEL = 'Soil', 'Building'",
    "",
    "",
]

_TAIL = [
    "",
    "",
    "def predict_soil_proba(frame):",
    "    # P(Soil) for each row. `frame` needs the seven SOURCE_BANDS as reflectance 0..1.",
    "    art = _load()",
    "    X = build_features(frame)[FEATURES]",
    "    if art['scaler'] is not None:",
    "        X = pd.DataFrame(art['scaler'].transform(X), columns=FEATURES, index=X.index)",
    "    d = xgb.DMatrix(X.astype(np.float32), feature_names=FEATURES)",
    "    return art['booster'].predict(d)",
    "",
    "",
    "def classify_soil_rows(frame):",
    "    # 'Soil' / 'Building' per row. Pass only the rows where ClassID == 5.",
    "    thr = _load()['meta'].get('threshold', THRESHOLD)",
    "    return np.where(predict_soil_proba(frame) >= thr, SOIL_LABEL, BUILDING_LABEL)",
    "",
]

NL = chr(10)
# FEATURE_SOURCE is the exact string section 3 exec'd to create the functions this model
# was trained with, so the shipped file cannot differ from the training code.
helper_text = NL.join(
    _HEADER + _CONST + FEATURE_SOURCE.strip().splitlines() + _TAIL) + NL

helper_path = art / "predict_soil_building.py"
helper_path.write_text(helper_text)
print(f"wrote {helper_path}  ({len(helper_text) / 1024:.1f} KB)")
print("-" * 78)
print(helper_text)

### 13a. End-to-end check: load from disk and reproduce the predictions

Nothing above is trusted here. The helper file is imported fresh, it loads the model and the
scaler off disk itself, and it is fed the **raw seven bands** for the test rows - no features
precomputed, no notebook state. If its labels match the ones section 10 scored, the artifact
directory is genuinely self-sufficient and can be copied into the repo as-is.

In [ ]:
import importlib.util

sys.dont_write_bytecode = True          # keep __pycache__ out of the artifact folder
spec = importlib.util.spec_from_file_location("predict_soil_building", helper_path)
mod = importlib.util.module_from_spec(spec)
# the helper resolves _MODEL_DIR relative to its own file, which is ART_DIR/soil_building_v1;
# here the artifacts sit in ART_DIR itself, so point it at the right place
spec.loader.exec_module(mod)
mod._MODEL_DIR = str(art)

n_check = min(200_000, len(te_idx))
raw_rows = df.iloc[te_idx[:n_check]][SOURCE_BANDS]     # raw bands only - nothing else

labels_helper = mod.classify_soil_rows(raw_rows)
labels_notebook = np.where(to_pred(p_test[:n_check]) == 1, "Soil", "Building")

agree = float(np.mean(labels_helper == labels_notebook))
print(f"rows checked            : {n_check:,}")
print(f"helper vs notebook agree: {agree:.6%}")

proba_helper = mod.predict_soil_proba(raw_rows)
print(f"max |proba difference|  : {np.abs(proba_helper - p_test[:n_check]).max():.3e}")

np.testing.assert_allclose(proba_helper, p_test[:n_check], rtol=1e-5, atol=1e-6)
assert agree == 1.0, "the generated helper does not reproduce the notebook's predictions"
print("\nOK - soil_building_v1/ reproduces these predictions from the raw bands alone.")
print("     Copy the folder next to capstone_model_v2/ and it is ready to wire in.")

import shutil
shutil.rmtree(art / "__pycache__", ignore_errors=True)

print("\nfinal contents:")
for p in sorted(art.iterdir()):
    print(f"  {p.name:<30}{p.stat().st_size / 1024:>10.1f} KB")

---
### Notes

* **The `ClassID == 5` filter is the point of this notebook, not a detail.** Section 2b prints
  the full crosstab before filtering and refuses to train if the filter leaves one label.
  Check that crosstab on the first run.

* **19 features, all derivable from seven bands.** That constraint comes from `EVALSCRIPT_S2`
  in `server/bimonthly_composite.py`, not from the data. If you ever widen the evalscript to
  fetch B05/B06/B07/B8A/B09, `BAND_NAMES` has to change in lockstep in four places
  (`server/bimonthly_composite.py`, `server/api_server.py`, `Capstone2.0/pipeline.py`) - and
  only then does retraining on more bands make sense.

* **`BSI_N`, not `BSI`.** The production pipeline computes `(B03+B08)/(B03-B08)` under the name
  `BSI`. That is not the Bare Soil Index and it is unbounded near `B03 == B08`. This model uses
  the standard normalised form under a different name so the two can never be confused.
  Section 3a asserts `BSI_N` matches the dataset's `BSI` column to 1e-4.

* **`SPLIT_STRATEGY`** defaults to `"spatial_block"` at 1 km. Switch it to `"stratified"` only
  to compare against `model-train-urban.ipynb`'s 0.9875 - that number is inflated by scoring on
  10 m neighbours of its own training pixels, and it is not what the map will do.

* **`best_model.json` vs `final_model.json`** - identical by construction. The first is written
  mid-training on every improvement (a killed session still leaves the best weights on disk);
  the second is the sliced booster written at the end. Section 8 asserts they agree.

* **Threshold** is tuned on validation, never on test. Section 10a then scores the model
  against the `NDBI > 0` rule it replaces, on the test set, and warns if it does not win.

* **Class imbalance** is handled with `scale_pos_weight` computed from the training split only.
  Set `BALANCE_CLASSES = False` to train unweighted.

* To download from Kaggle: `/kaggle/working/` is saved as the run's output - take the whole
  `soil_building_v1/` folder from the Output tab.